## Real Data in Databricks Free Edition

You can work with real data in Databricks Free Edition, but the options are more limited than in a paid workspace. The main realistic path is **file-based streaming**, not live external systems like Kafka.

### What Doesn't Work in Free Edition

Connecting to external streaming services like **Apache Kafka** is not supported in the serverless Free Edition. If you try, you will see errors like:

- `INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED`
- `TEMP_CHECKPOINT_LOCATION_NOT_SUPPORTED`

### What Does Work: File-Based Streaming

Use **Auto Loader** to stream data from files stored in **Unity Catalog Volumes**. This is the closest you can get to practicing with "real" streaming data, because new files that appear in a volume are picked up as new data.

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.streaming_data;

In [0]:
#use spark.readStream to read from that volume using Auto Loader. The schemaLocation option is important for schema inference

# Define paths
source_path = "/Volumes/workspace/default/streaming_data"
checkpoint_path = "/Volumes/workspace/default/checkpoints/wikipedia_stream"

# Read the stream using Auto Loader
raw_df = (spark.readStream
    .format("cloudFiles")  # Auto Loader (File Ingestion)
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .load(source_path)
)
#If you are loading from a Delta table, you are using standard Structured Streaming, not Auto Loader.
# Auto Loader (.format("cloudFiles")) is exclusively for reading raw files (like JSON, CSV, or Parquet) from a cloud storage folder (S3, ADLS) where new files are constantly arriving.

# Write to a Delta table
(raw_df.writeStream
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("workspace.default.wikipedia_stream_output")
)

**"cloudFiles.schemaLocation"**
```
 remembers the Schema. in first run it read some files workout on schema and save it in schema location,
  in next run it uses the saved schema instead of guessing it making it faster and reliable
 ```

**"checkpointLocation"**
```
remember progress (which files are done)
```



## Watermarks + Windowed Aggregations

### Why It Matters

- Real-world data arrives late (network delays, mobile devices offline)
- You often need "last 5 minutes" or "hourly" aggregations
- Without watermarks, streaming state grows forever and memory explodes

### The Concept in 30 Seconds

```python
# Windowed aggregation with watermark
(df
    .withWatermark("event_time", "10 minutes")    # wait up to 10 min for late data
    .groupBy(window("event_time", "5 minutes"))   # 5-minute tumbling windows
    .count()
)
```

| Term | Meaning |
|---|---|
| **Window** | Time bucket (every 5 minutes) |
| **Watermark** | How long to wait for late data (10 minutes) |


##Revision incremental load
**There’s no special setting that marks a column as the watermark. You (the engineer) choose it after looking at the source table.**

```
from pyspark.sql import functions as F
last = spark.sql('select max(updated_at) from workspace.default.gold_table').first()[0]

df = spark.table('select * from workspace.default.bronze_table')
if last is not None:
    df = df.filter(F.col('updated_at')>F.lit(last))

df.write.format('delta').mode('append').saveAsTable('workspace.default.gold_table')
```